|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Against the roof<h1>|
|<h2>Lecture:</h2>|<h1><b>Code challenge: measure your engine against the roof<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

Measure an engine against physics, in units that travel between cards.

Stage 25 gates the capstone on one ratio: the floor of the steps that the
engine ran, divided by the time that they took. Build that floor, use it to
compare models, and then measure one real step against it.

In [ ]:
# Published configs. No download: the formula needs only these numbers.
MODELS = {
    'Qwen3-0.6B':   dict(layers=28, hidden=1024, heads=16, kv_heads=8, head_dim=128,
                         intermediate=3072, vocab=151936),
    'Qwen3-1.7B':   dict(layers=28, hidden=2048, heads=16, kv_heads=8, head_dim=128,
                         intermediate=6144, vocab=151936),
    'Llama-3.2-1B': dict(layers=16, hidden=2048, heads=32, kv_heads=8, head_dim=64,
                         intermediate=8192, vocab=128256),
    'Qwen3-4B':     dict(layers=36, hidden=2560, heads=32, kv_heads=8, head_dim=128,
                         intermediate=9728, vocab=151936),
    'Llama-3.1-8B': dict(layers=32, hidden=4096, heads=32, kv_heads=8, head_dim=128,
                         intermediate=14336, vocab=128256),
}

def model_sizes(config, bytes_per_weight=2, bytes_per_kv=2):
    """-> (bytes that one decode step reads, parameters, KV bytes for each token)."""
    attention = (2 * config['hidden'] * config['heads'] * config['head_dim']       # q, o
                 + 2 * config['hidden'] * config['kv_heads'] * config['head_dim'])  # k, v
    mlp = 3 * config['hidden'] * config['intermediate']
    params = (config['layers'] * (attention + mlp)
              + config['vocab'] * config['hidden'])        # decode reads the lm_head
    kv_bytes = 2 * config['layers'] * config['kv_heads'] * config['head_dim'] * bytes_per_kv
    return params * bytes_per_weight, params, kv_bytes

for name, config in MODELS.items():
    weight_bytes, params, kv_bytes = model_sizes(config)
    print(f'{name:13} reads {weight_bytes/1e9:5.2f} GB for each step, '
          f'KV {kv_bytes/1024:4.0f} KiB for each token')

# Exercise 1: the floor of one step

In [ ]:
def step_floor(weight_bytes, kv_bytes, params, num_tokens, context_tokens, bandwidth, flops):
    """Seconds. The weight read or the arithmetic, the larger of the two.
    Then the KV reads."""
    return (max(weight_bytes / bandwidth, 2 * params * num_tokens / flops)
            + context_tokens * kv_bytes / bandwidth)

assert step_floor(1e9, 1e5, 5e8, 1, 0, 1e11, 1e14) == 1e9 / 1e11
assert step_floor(1e9, 1e5, 5e8, 10_000, 0, 1e11, 1e14) == 2 * 5e8 * 1e4 / 1e14
print('the floor is correct')

# Exercise 2: the ceiling, in tokens for each GB/s

The ceiling in tok/s is (tokens in the step) / (floor). Divide it by the read
bandwidth. It then no longer depends on the card, as long as the weights, and
not the arithmetic, set the max. Compute it for the four models at batch 1, and
at 32 sequences of 4k context.

In [ ]:
def tokens_per_gb(weight_bytes, kv_bytes, batch, context):
    """The ceiling of tok/s, divided by the read bandwidth in GB/s. Below the
    ridge the weight read is the larger term, so FLOPS is not in it."""
    return batch / ((weight_bytes + batch * context * kv_bytes) / 1e9)

print(f'{"model":13} {"batch 1":>9} {"32 x 4k ctx":>12}   (tokens for each GB/s)')
for name, config in MODELS.items():
    weight_bytes, _, kv_bytes = model_sizes(config)
    batch_one = tokens_per_gb(weight_bytes, kv_bytes, 1, 0)
    long_context = tokens_per_gb(weight_bytes, kv_bytes, 32, 4096)
    print(f'{name:13} {batch_one:>9.2f} {long_context:>12.2f}')

# Exercise 3: one real step against its floor

A batch-1 decode step of the capstone model, eager and captured in a graph.
The floor comes from the bandwidth and the FLOPS that this card has now.

In [ ]:
class StaticBackend:
    """A dense cache that a CUDA graph can capture. Each layer has a fixed
    (batch, kv_heads, max_len, head_dim) buffer, and a (batch,) tensor holds
    the positions. Each row of the batch is one sequence. A step does no
    allocation and no host sync."""
    def __init__(self, config, batch, max_len):
        shape = (batch, config.num_kv_heads, max_len, config.head_dim)
        self.key_cache = [torch.zeros(shape, dtype=config.dtype, device='cuda')
                          for _ in range(config.num_layers)]
        self.value_cache = [torch.zeros_like(cache) for cache in self.key_cache]
        self.positions = torch.zeros(batch, dtype=torch.long, device='cuda')
        self.batch_rows = torch.arange(batch, device='cuda')
        self.key_positions = torch.arange(max_len, device='cuda')

    def __call__(self, layer, query, key, value):
        batch, num_heads, head_dim = query.shape
        keys, values = self.key_cache[layer], self.value_cache[layer]
        keys[self.batch_rows, :, self.positions] = key
        values[self.batch_rows, :, self.positions] = value
        num_kv_heads = keys.shape[1]
        # Two grouped matmuls. SDPA with a mask and GQA uses a path that
        # copies K and V, and that copy costs more than the whole step.
        grouped_query = query.view(batch, num_kv_heads, num_heads // num_kv_heads, head_dim)
        scores = torch.matmul(grouped_query, keys.transpose(-1, -2)) * head_dim ** -0.5
        visible = self.key_positions[None, :] <= self.positions[:, None]
        scores = scores.masked_fill(~visible[:, None, None, :], float('-inf'))
        output = torch.matmul(scores.softmax(-1).to(values.dtype), values)
        return output.reshape(batch, num_heads * head_dim)

def graphed(step):
    """Capture step() one time. -> a function that replays it."""
    side_stream = torch.cuda.Stream()
    side_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side_stream):
        for _ in range(3):
            step()
    torch.cuda.current_stream().wait_stream(side_stream)
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        output = step()
    def replay():
        graph.replay()
        return output
    return replay

from tvllm import load_model
model = load_model('Qwen/Qwen3-1.7B')
config = model.config

def decode_step(backend, batch):
    """-> a function that runs one decode step of the batch on backend."""
    token_ids = torch.zeros(batch, dtype=torch.long, device='cuda')
    logits_rows = torch.arange(batch, device='cuda')
    return lambda: model.forward(token_ids, backend.positions, backend, logits_rows)

POSITION = 40
backend = StaticBackend(config, batch=1, max_len=256)
backend.positions.fill_(POSITION)
step = decode_step(backend, batch=1)
bandwidth, flops = cudalib.read_bandwidth(), cudalib.matmul_flops()
eager_ms = cudalib.bench_ms(step, iters=30, best_of=3)
graph_ms = cudalib.bench_ms(graphed(step), iters=30, best_of=3)
weight_bytes = model.weight_bytes()
params = weight_bytes / 2                    # bf16: 2 bytes for each parameter
floor_ms = step_floor(weight_bytes, model.kv_bytes_per_token(), params,
                      1, POSITION + 1, bandwidth, flops) * 1e3
print(f'eager reaches {100 * floor_ms / eager_ms:.0f}% of the floor, '
      f'graphed reaches {100 * floor_ms / graph_ms:.0f}%')

# Exercise 4: fewer bytes

Stage 18b makes the weights int8. An FP8 KV cache makes the other large read
one byte for each value. Compute what each one does to the ceiling, as a
ratio against bf16.

In [ ]:
print(f'{"model":13} {"int8 weights, batch 1":>22} {"fp8 KV, 32 x 4k":>16}')
for name, config in MODELS.items():
    weight_bytes, _, kv_bytes = model_sizes(config)
    int8_weight_bytes, _, _ = model_sizes(config, bytes_per_weight=1)
    _, _, fp8_kv_bytes = model_sizes(config, bytes_per_kv=1)
    int8_gain = weight_bytes / int8_weight_bytes
    fp8_gain = (tokens_per_gb(weight_bytes, fp8_kv_bytes, 32, 4096)
                / tokens_per_gb(weight_bytes, kv_bytes, 32, 4096))
    print(f'{name:13} {int8_gain:>21.2f}x {fp8_gain:>15.2f}x')

### What the four exercises show

**The floor has two terms, and the batch picks which one matters.** At batch
1, the weight read is everything, and the smallest model wins. At 32 x 4k, the
KV read is most of the step, and the model with the fewest KV bytes for each
token wins. Llama-3.2-1B beats Qwen3-0.6B there, because its KV is 32 KiB for
each token against 112 KiB.

**A large weight read hides the small costs.** Qwen3-1.7B reads 3.4 GB in
each step. That read hides most of the CPU issue time and most of the small
kernels, so the step comes close to its floor. A ratio a little above 100% is
not a fault. The floor uses the bandwidth of a simple copy kernel, and a tuned
matmul can read a little faster.

On a smaller model the read is shorter, and it hides less. The graphed step
removes the CPU issue time. What remains is about a thousand small kernels,
each of which moves few bytes. Fusion (for example of RoPE with the Q and K
norms) attacks that.

**Quantization attacks the term that dominates.** Int8 weights halve the first
term, so they help at batch 1. FP8 KV halves the second term, so it helps at
long context. Neither ratio depends on the card.

    ./vc guide 25